# PR18 · Fit a tensor and interpret FA and MD within its limits

<!-- paper-first -->
### Research question

**Reading:** [PP06](../../curriculum/papers/processing.md#pp06). Review the assigned figure or result before starting the lesson.

**Question:** Why does fitting one diffusion tensor not establish the number or course of axonal bundles?

Record a prediction, a source location, and one point you want this lesson to clarify. Ask your AI tutor to distinguish the paper’s evidence from its interpretation.
<!-- /paper-first -->

**Format:** 75–100 minutes for this lesson and its small executable lab, followed by the explicitly labeled upstream practice. Run this notebook from a fresh kernel, top to bottom. Core data are synthetic. Software: NumPy, SciPy, Matplotlib; extra imports are stated in code. This is one stage of a longer processing course, not a replacement for supervised research training.

## Learning objectives

- Derive a six-parameter tensor fit from attenuation.
- Calculate eigenvalues, mean diffusivity and fractional anisotropy with units.
- Detect a b-value unit error that FA alone misses.

## Understand the operation

The diffusion tensor approximates local diffusion behavior with a symmetric3×3 matrix. Under its signal model, S/S0=exp(−b gᵀDg). The six independent elements can be fitted from appropriately distributed diffusion directions and a reference signal. Six equations are only a mathematical minimum; noisy acquisitions need more information and suitable estimation. Direction coverage, b-values, outliers and signal-floor effects affect stability.

The eigenvectors describe principal axes of the fitted ellipsoid, with an arbitrary sign for each vector. Eigenvalues have diffusivity units. Mean diffusivity is their mean. Fractional anisotropy summarizes their relative spread and is dimensionless. Multiplying every eigenvalue by the same positive factor leaves FA unchanged while changing MD. This means a plausible FA map does not validate b-value units. Negative fitted eigenvalues can indicate fitting/model problems; clipping them is a consequential modeling choice, not a substitute for investigating the data.

A single tensor cannot resolve multiple crossing fiber populations within a voxel. Lower FA can reflect crossing geometry, partial volume, noise or pathology among other causes. It is not a specific assay of axon number, myelin content or “white-matter health.” The principal eigenvector represents an undirected model orientation, not the direction of neural signaling.

This lab fits a noise-free synthetic tensor by log-linear least squares, reconstructs its signal and computes FA/MD. The exact recovery is a unit test of the model algebra under ideal data, not expected real-world accuracy. The next notebook deliberately violates the single-tensor assumption with crossing components.

## Transformation contract

**Input:** S0, diffusion-weighted signal, b-values and unit directions. **Output:** tensor, reconstructed attenuation and scalar summaries. **Preserved by ideal fit:** the signal under this exact model. **Compressed/lost:** deviations not representable by one tensor; FA also discards magnitude scale and orientation.


## Read the actual course material

- [DIPY: diffusion tensor reconstruction](https://github.com/dipy/dipy/blob/05df74a36c48e38ef1aa7420440abb9f1d14e6d4/doc/examples/reconst_dti.py). BSD-3-Clause project; linked source, no example code copied.
- [Oxford FSL: FDT pipeline, TOPUP, EDDY, DTIFIT and TBSS](https://pages.fmrib.ox.ac.uk/fslcourse/practicals/fdt1/index.html). Public university practical; educational data terms at source; linked only.

Read the named topic alongside this lesson; compare its real-image assumptions with our controlled example. These notebooks use original explanations and original code, not copied upstream passages. The source chapter is the place to continue to a complete real-tool practical. External software and downloaded datasets are not silently run by this notebook.


## Predict, then ask your AI assistant

Use Goose with your installed Ollama model, or ChatGPT. The model is a tutor and code author; the local Python runtime performs these calculations. Paste:

> Write the design matrix from the six tensor terms and verify its rank. Fit the known tensor, reconstruct the signal, and show how the wrong b-value scale changes MD but not FA. State units and avoid attributing a unique biological cause to FA. Return at most 20 executable lines per cell, show units and array shapes, and preserve the original. Explain the prediction before running. If an assertion fails, diagnose the disagreement rather than deleting the check.

Write your prediction before executing the reference cells below.


In [1]:
import numpy as np
rng=np.random.default_rng(1818); g=rng.normal(size=(60,3)); g/=np.linalg.norm(g,axis=1)[:,None]
b=1000.; D=np.diag([.0017,.0003,.0003]); S0=1000.
S=S0*np.exp(-b*np.einsum('ni,ij,nj->n',g,D,g))
gx,gy,gz=g.T
X=np.column_stack([gx*gx,gy*gy,gz*gz,2*gx*gy,2*gx*gz,2*gy*gz])
y=-np.log(S/S0)/b
beta=np.linalg.lstsq(X,y,rcond=None)[0]
fit=np.array([[beta[0],beta[3],beta[4]],[beta[3],beta[1],beta[5]],[beta[4],beta[5],beta[2]]])
eigenvalues=np.linalg.eigvalsh(fit)
FA=lambda e: np.sqrt(1.5*np.sum((e-e.mean())**2)/np.sum(e**2))
print('rank, eigenvalues, MD, FA:',np.linalg.matrix_rank(X),eigenvalues,eigenvalues.mean(),FA(eigenvalues))
assert np.linalg.matrix_rank(X)==6 and np.allclose(fit,D,atol=1e-12)
assert np.isclose(eigenvalues.mean(),.0023/3) and .7<FA(eigenvalues)<1


rank, eigenvalues, MD, FA: 6 [0.0003 0.0003 0.0017] 0.0007666666666666658 0.7990222037494897


In [2]:
bad_beta=np.linalg.lstsq(X,-np.log(S/S0)/1.,rcond=None)[0]
bad_fit=np.array([[bad_beta[0],bad_beta[3],bad_beta[4]],
                  [bad_beta[3],bad_beta[1],bad_beta[5]],
                  [bad_beta[4],bad_beta[5],bad_beta[2]]])
bad_eigenvalues=np.linalg.eigvalsh(bad_fit)
assert np.isclose(FA(eigenvalues),FA(bad_eigenvalues))
assert np.isclose(bad_eigenvalues.mean()/eigenvalues.mean(),1000)
reconstructed=S0*np.exp(-b*np.einsum('ni,ij,nj->n',g,fit,g))
assert np.allclose(S,reconstructed)
print('Wrong b units: FA unchanged; MD multiplied by1000.')


Wrong b units: FA unchanged; MD multiplied by1000.


## Check and explain

The rank is6, eigenvalues recover[.0003,.0003,.0017]mm²/s, MD is about.0007667mm²/s, and FA is about.799. Dividing by b=1 instead of1000 makes MD1000 times larger while leaving FA unchanged.

## Deliberately wrong method

Checking only that FA lies between0 and1 misses the unit error. Calling the principal eigenvector a directed connection or treating low FA as a unique disease marker overinterprets the tensor representation.

## Transfer to an actual dataset or tool — guided assignment

Follow the pinned DIPY tensor reconstruction example on its separately obtained data or Oxford DTIFIT using corrected DWI and rotated gradients. Inspect S0 coverage, fitted residuals, eigenvalues and orientation in well-understood regions. Record the fit method and exact b-value units. Use the documented returned fields rather than asking the AI to invent property names. TBSS/group statistics are additional external modules, not a consequence of fitting one subject.

**Submit:** a transformation card, one labeled figure or numerical result, the failed-method diagnosis, and the upstream-practice evidence. If the external exercise has not been run, mark it **not executed** and state the missing software/data; do not convert a proposed command into a claimed result.

## Exit questions and answer key

1. Why is FA unchanged by global eigenvalue scaling? **Answer:** its numerator and denominator scale equally; it describes relative anisotropy.
2. What does poor reconstruction indicate? **Answer:** possible noise, metadata, outlier or model-mismatch problems; it does not identify one cause automatically.


### Return to the research question

Revisit [PP06](../../curriculum/papers/processing.md#pp06) and your initial prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure or section locator. Which part of the published result remains open after this exercise?
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Include this entry in the A2 portfolio when relevant.
